In [1]:
import numpy as np
import pandas as pd
from plotly.io import show

from skfolio import Population, RiskMeasure
from skfolio.datasets import load_factors_dataset, load_sp500_dataset
from skfolio.distribution import VineCopula
from skfolio.measures import (
    correlation,
    cvar,
    kurtosis,
    mean,
    skew,
    standard_deviation,
    value_at_risk,
)
from skfolio.optimization import HierarchicalRiskParity, RiskBudgeting
from skfolio.preprocessing import prices_to_returns
from skfolio.prior import EntropyPooling, FactorModel, SyntheticData
from skfolio.utils.figure import plot_kde_distributions

# Load stock price and factor data
prices = load_sp500_dataset()
prices = prices[["AMD", "BAC", "GE", "JNJ", "JPM", "LLY", "PG"]]
factor_prices = load_factors_dataset()

# Convert to daily returns
X, factors = prices_to_returns(prices, factor_prices)

print("Shapes:")
print(f"X: {X.shape}")
print(f"factors: {factors.shape}")

print(X.tail())
print(factors.tail())

Shapes:
X: (2263, 7)
factors: (2263, 5)
                 AMD       BAC        GE       JNJ       JPM       LLY  \
Date                                                                     
2022-12-21  0.040430  0.015223  0.033001  0.011444  0.011248  0.023275   
2022-12-22 -0.056442 -0.008848 -0.014582 -0.003655 -0.011355 -0.007339   
2022-12-23  0.010335  0.002443  0.000235  0.002539  0.004749  0.007090   
2022-12-27 -0.019374  0.001875  0.012849 -0.000280  0.003504 -0.008208   
2022-12-28 -0.011064  0.007360 -0.010502 -0.004341  0.005463  0.000932   

                  PG  
Date                  
2022-12-21  0.009170  
2022-12-22  0.002308  
2022-12-23  0.002825  
2022-12-27  0.008713  
2022-12-28 -0.012926  
                MTUM      QUAL      SIZE      USMV      VLUE
Date                                                        
2022-12-21  0.014312  0.017884  0.014371  0.012005  0.013246
2022-12-22 -0.010977 -0.015411 -0.012070 -0.007315 -0.011989
2022-12-23  0.010897  0.005889  0.00

In [2]:
def summary(X: pd.DataFrame, sample_weight: np.ndarray | None = None) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "Mean": mean(X, sample_weight=sample_weight),
            "Volatility": standard_deviation(X, sample_weight=sample_weight),
            "Skew": skew(X, sample_weight=sample_weight),
            "Kurtosis": kurtosis(X, sample_weight=sample_weight),
            "VaR at 90%": value_at_risk(X, beta=0.90, sample_weight=sample_weight),
            "CVaR at 90%": cvar(X, beta=0.90, sample_weight=sample_weight),
        }
    )


print(f"Corr(BAC, JPM): {correlation(X[['BAC', 'JPM']])[0][1]:.2%}")
summary(X)

Corr(BAC, JPM): 90.87%


,Mean,Volatility,Skew,Kurtosis,VaR at 90%,CVaR at 90%
AMD,0.001902,0.037314,1.323839,22.604470,0.035782,0.061211
BAC,0.000581,0.019822,0.283198,13.312827,0.020279,0.034383
GE,-0.000099,0.021970,0.179690,9.851668,0.021926,0.039269
JNJ,0.000466,0.011443,-0.261929,12.610478,0.011002,0.020188
JPM,0.000621,0.017354,0.343518,17.000055,0.016999,0.029650
LLY,0.001102,0.016708,0.910982,14.984163,0.015920,0.027036
PG,0.000465,0.011699,0.261102,16.276273,0.010843,0.020274


In [3]:
groups = {
    "AMD": ["Technology", "Growth"],
    "BAC": ["Financials", "Value"],
    "GE": ["Industrials", "Value"],
    "JNJ": ["Healthcare", "Defensive"],
    "JPM": ["Financials", "Income"],
    "LLY": ["Healthcare", "Defensive"],
    "PG": ["Consumer", "Defensive"],
}

entropy_pooling = EntropyPooling(
    mean_views=[
        "JPM == -0.002",
        "PG >= LLY",
        "BAC >= prior(BAC) * 1.2",
        "Financials == 2 * Growth",
    ],
    variance_views=[
        "BAC == prior(BAC) * 4",
    ],
    correlation_views=[
        "(BAC,JPM) == 0.80",
        "(BAC,JNJ) <= prior(BAC,JNJ) * 0.5",
    ],
    skew_views=[
        "BAC == -0.05",
    ],
    cvar_views=[
        "GE == 0.07",
    ],
    cvar_beta=0.90,
    groups=groups,
)
entropy_pooling.fit(X)

print(f"Relative Entropy : {entropy_pooling.relative_entropy_:.2f}")
print(
    f"Effective Number of Scenarios : {entropy_pooling.effective_number_of_scenarios_:.0f}"
)

Relative Entropy : 0.67
Effective Number of Scenarios : 1153


In [4]:
sample_weight = entropy_pooling.return_distribution_.sample_weight

print(f"sample_weight Shape: {sample_weight.shape}")
print(
    f"Corr(BAC, JPM): {correlation(X[['BAC', 'JPM']], sample_weight=sample_weight)[0][1]:.2%}"
)
summary(X, sample_weight=sample_weight)

sample_weight Shape: (2263,)
Corr(BAC, JPM): 80.00%


,Mean,Volatility,Skew,Kurtosis,VaR at 90%,CVaR at 90%
AMD,-0.000651,0.045605,2.860409,33.865365,0.042781,0.071557
BAC,0.000697,0.039777,-0.050005,6.756438,0.054128,0.072767
GE,-0.001392,0.032328,-0.897156,8.318331,0.033021,0.070000
JNJ,0.000099,0.017201,-1.324925,14.924965,0.011936,0.030203
JPM,-0.002000,0.035507,-0.654309,10.228875,0.027399,0.076548
LLY,0.000807,0.018280,0.721293,14.246732,0.016299,0.031244
PG,0.000807,0.015036,1.006576,17.105880,0.010674,0.024827


In [5]:
fig = plot_kde_distributions(
    X,
    sample_weight=sample_weight,
    percentile_cutoff=0.1,
    title="Distribution of Asset Returns (Prior vs. Posterior)",
    unweighted_suffix="Prior",
    weighted_suffix="Posterior",
)
show(fig)

In [6]:
bench = RiskBudgeting(risk_measure=RiskMeasure.CVAR, cvar_beta=0.9)
model = RiskBudgeting(
    risk_measure=RiskMeasure.CVAR, cvar_beta=0.9, prior_estimator=entropy_pooling
)

bench.fit(X)
model.fit(X)

print(bench.weights_)
print(model.weights_)

[0.07696053 0.10666039 0.11098278 0.19798089 0.11930962 0.16615805
 0.22194774]
[0.08274408 0.08632226 0.07945493 0.18252283 0.06747649 0.22682331
 0.27465609]


In [7]:
sample_weight = model.prior_estimator_.return_distribution_.sample_weight

portfolio_bench = bench.predict(X)
portfolio_bench.name = "Benchmark (Optimized on prior)"
portfolio_ep = model.predict(X)
portfolio_ep.name = "Optimized on EP posterior"

In [8]:
population = Population([portfolio_bench, portfolio_ep])
population.plot_contribution(measure=RiskMeasure.CVAR)

In [9]:
population.set_portfolio_params(sample_weight=sample_weight)
population.plot_contribution(measure=RiskMeasure.CVAR)

In [10]:
factor_entropy_pooling = EntropyPooling(mean_views=["QUAL == 0.0005"])

factor_model = FactorModel(factor_prior_estimator=factor_entropy_pooling)

model = RiskBudgeting(risk_measure=RiskMeasure.CVAR, prior_estimator=factor_model)

model.fit(X, factors)
print(model.weights_)

sample_weight = model.prior_estimator_.return_distribution_.sample_weight
summary(factors, sample_weight)

[0.08586464 0.10101143 0.11469179 0.20548591 0.11275494 0.17553009
 0.20466121]


,Mean,Volatility,Skew,Kurtosis,VaR at 90%,CVaR at 90%
MTUM,0.000587,0.012714,-0.380000,13.542578,0.012760,0.023557
QUAL,0.000500,0.011505,-0.200278,15.466345,0.011181,0.021009
SIZE,0.000488,0.011610,-0.819663,22.174947,0.010762,0.020940
USMV,0.000485,0.009488,-0.510849,22.217140,0.008825,0.016744
VLUE,0.000418,0.012393,-0.580626,18.171558,0.011946,0.022327


In [11]:
vine = VineCopula(log_transform=True, n_jobs=-1, random_state=0)

factor_synth = SyntheticData(n_samples=100_000, distribution_estimator=vine)

factor_entropy_pooling = EntropyPooling(
    prior_estimator=factor_synth,
    cvar_beta=0.9,
    cvar_views=["QUAL == 0.10"],
)

factor_model = FactorModel(factor_prior_estimator=factor_entropy_pooling)

model = HierarchicalRiskParity(
    risk_measure=RiskMeasure.CVAR, prior_estimator=factor_model
)

model.fit(X, factors)
print(model.weights_)

[0.04419332 0.08634233 0.09537424 0.17007835 0.09514572 0.13505241
 0.37381362]


In [12]:
model = HierarchicalRiskParity(risk_measure=RiskMeasure.CVAR)

model.fit(X)
print(model.weights_)

portfolio = model.predict(X)
portfolio.name = "HRP Unstressed"

# Add to a Population for better comparison with the stressed portfolios.
population = Population([portfolio])

[0.06021919 0.1059091  0.08763318 0.18841288 0.11675892 0.25472467
 0.18634206]


In [13]:
vine = VineCopula(log_transform=True, n_jobs=-1, random_state=0)

synth = SyntheticData(n_samples=100_000, distribution_estimator=vine)

entropy_pooling = EntropyPooling(
    prior_estimator=synth, cvar_beta=0.90, cvar_views=["AMD == 0.10"]
)

entropy_pooling.fit(X)

# We retrieve the stressed distribution:
stressed_dist = entropy_pooling.return_distribution_

# We stress-test our portfolio:
stressed_ptf = model.predict(stressed_dist)

# Add the stressed portfolio to the population
stressed_ptf.name = "HRP Stressed"
population.append(stressed_ptf)

In [14]:
factor_synth = SyntheticData(n_samples=100_000, distribution_estimator=vine)

factor_entropy_pooling = EntropyPooling(
    prior_estimator=factor_synth,
    cvar_beta=0.90,
    cvar_views=["QUAL == 0.10"],
)

factor_model = FactorModel(factor_prior_estimator=factor_entropy_pooling)

factor_model.fit(X, factors)

# We retrieve the stressed distribution:
stressed_dist = factor_model.return_distribution_

# We stress-test our portfolio:
stressed_ptf = model.predict(stressed_dist)

# Add the stressed portfolio to the population
stressed_ptf.name = "HRP Factor Stressed"
population.append(stressed_ptf)

In [15]:
pop_summary = population.summary()
pop_summary.loc[
    [
        "Mean",
        "Standard Deviation",
        "CVaR at 95%",
        "Annualized Sharpe Ratio",
        "Worst Realization",
    ]
]

,HRP Unstressed,HRP Stressed,HRP Factor Stressed
Mean,0.070%,-0.030%,-0.55%
Standard Deviation,1.14%,1.40%,2.99%
CVaR at 95%,2.64%,3.70%,11.67%
Annualized Sharpe Ratio,0.96,-0.34,-2.91
Worst Realization,9.16%,23.95%,17.16%


In [16]:
population.plot_returns_distribution(percentile_cutoff=0.0500)

conclusion:
demonstrated how to leverage Entropy Pooling to integrate views into every stage of portfolio management, from ex-ante optimization to ex-post stress testing.